## 🎯 Casos Prácticos: Features H3 y Temporales

### 🗺️ Caso 1: Interpretar Features H3

**Problema**: Feature H3 encoded es importante, pero ¿qué zonas contribuyen más?

**Solución con SHAP**:

```python
import shap
import pandas as pd
import h3

# Entrenar modelo
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

# Analizar feature H3
h3_col = 'zona_h3_encoded'
h3_idx = X_test.columns.tolist().index(h3_col)

# Top zonas por SHAP value promedio
df_h3_shap = pd.DataFrame({
    'h3_encoded': X_test[h3_col],
    'h3_original': X_test['zona_h3'],  # Sin encodear
    'shap_value': shap_values.values[:, h3_idx]
})

top_zonas = df_h3_shap.groupby('h3_original')['shap_value'].mean().sort_values(ascending=False)

print("Top 10 zonas H3 por impacto positivo:")
for h3_hex, shap_val in top_zonas.head(10).items():
    lat, lon = h3.h3_to_geo(h3_hex)
    print(f"  {h3_hex}: SHAP = {shap_val:+.2f}  (lat={lat:.4f}, lon={lon:.4f})")

print("\nTop 10 zonas H3 por impacto negativo:")
for h3_hex, shap_val in top_zonas.tail(10).items():
    lat, lon = h3.h3_to_geo(h3_hex)
    print(f"  {h3_hex}: SHAP = {shap_val:+.2f}  (lat={lat:.4f}, lon={lon:.4f})")
```

💡 **Insight**: Identificar zonas geográficas con mayor impacto en ventas.

---

### 📅 Caso 2: Interpretar Features Temporales

**Problema**: ¿Cómo interactúan día de semana y mes?

**Solución con SHAP Dependence Plot**:

```python
# Dependence plot con interacción
shap.dependence_plot(
    'dia_semana',
    shap_values.values,
    X_test,
    interaction_index='mes'
)
plt.title('Interacción: Día de Semana x Mes')
plt.show()
```

**Insights típicos**:
- Viernes tiene mayor impacto en diciembre (temporada alta)
- Lunes tiene impacto similar todo el año
- Fin de semana es más importante en verano

---

### 🔄 Caso 3: Explicar Predicción Anómala

**Problema**: Predicción muy alta (¿por qué?).

**Solución con SHAP Waterfall**:

```python
# Encontrar predicciones altas
predictions = model.predict(X_test)
high_preds = predictions.argsort()[-5:]  # Top 5 predicciones

# Explicar cada una
for i in high_preds:
    print(f"\nPredicción #{i}: ${predictions[i]:.2f}")
    shap.plots.waterfall(shap_values[i])
```

**Interpretación**:
```
Predicción #542: $280.50

E[f(X)] = $120.00
  ├─ zona_h3 = microcentro  +$45.00  ⬆️
  ├─ dia_semana = viernes   +$35.00  ⬆️
  ├─ mes = diciembre        +$50.00  ⬆️  (Navidad!)
  ├─ segmento = premium     +$25.00  ⬆️
  └─ cliente_frecuente      +$5.50   ⬆️
f(x) = $280.50
```

💡 **Insight**: Predicción alta debido a **múltiples factores positivos** alineados (zona premium + viernes + diciembre).

---

## ✅ Mejores Prácticas de Interpretabilidad

### ✅ DO: Buenas Prácticas

#### 1. **Siempre Usar Test Set para Interpretabilidad**
```python
# ✅ BIEN: Calcular SHAP en test
shap_values = explainer(X_test)

# ❌ MAL: Calcular SHAP en train (sobreajustado)
shap_values = explainer(X_train)
```

#### 2. **Validar con Múltiples Métodos**
```python
# ✅ BIEN: Comparar Feature Importance, Permutation y SHAP
fi = model.feature_importances_
pi = permutation_importance(model, X_test, y_test)
shap_global = np.abs(shap_values.values).mean(axis=0)

# ¿Coinciden? Si no, investigar por qué
```

#### 3. **Documentar Explicaciones**
```python
# ✅ BIEN: Guardar explicaciones
import pickle

with open('shap_values.pkl', 'wb') as f:
    pickle.dump(shap_values, f)

# Guardar imágenes
shap.summary_plot(shap_values.values, X_test, show=False)
plt.savefig('shap_summary.png', dpi=300, bbox_inches='tight')
```

#### 4. **Explicar Predicciones Críticas**
```python
# ✅ BIEN: Explicar decisiones importantes
# - Predicciones altas (oportunidades)
# - Predicciones bajas (riesgos)
# - Predicciones anómalas (outliers)

high_pred_idx = predictions.argsort()[-10:]  # Top 10
for i in high_pred_idx:
    shap.plots.waterfall(shap_values[i])
```

#### 5. **Comunicar a Audiencias Diferentes**
```python
# Para data scientists: SHAP summary plots, dependence plots
# Para negocio: PDP (más simples), waterfall para casos específicos
# Para usuarios finales: Waterfall con lenguaje natural
```

---

### ❌ DON'T: Errores Comunes

#### 1. **NO Confiar Solo en Feature Importance**
```python
# ❌ MAL: Solo feature importance (sesgado)
importances = model.feature_importances_

# ✅ BIEN: Validar con Permutation o SHAP
```

#### 2. **NO Interpretar Features Enco ded Sin Contexto**
```python
# ❌ MAL: "zona_h3_encoded = 42 tiene SHAP = +$20"
# ¿Qué zona es 42?

# ✅ BIEN: Mapear a valores originales
zone_mapping = dict(zip(df['zona_h3_encoded'], df['zona_h3']))
print(f"Zona {42} = {zone_mapping[42]}")
```

#### 3. **NO Asumir Causalidad**
```python
# ❌ MAL: "Si cambiamos zona H3 → ventas subirán $20"
# Interpretabilidad != Causalidad

# ✅ BIEN: "Zona H3 está ASOCIADA con +$20 en ventas"
```

#### 4. **NO Ignorar Interacciones**
```python
# ❌ MAL: Analizar features aisladamente

# ✅ BIEN: Usar SHAP dependence con interaction_index
shap.dependence_plot('dia_semana', shap_values.values, X_test,
                      interaction_index='mes')
```

#### 5. **NO Olvidar Incertidumbre**
```python
# ❌ MAL: "Este feature aporta exactamente +$15.23"

# ✅ BIEN: "Este feature aporta aproximadamente +$15 (±$2)"
# Reportar intervalos de confianza
```

---

### 📊 Checklist de Interpretabilidad

Antes de presentar un modelo:

- [ ] Calculaste feature importance (si tree-based)
- [ ] Validaste con permutation importance
- [ ] Generaste SHAP summary plot
- [ ] Analizaste PDP para top 5 features
- [ ] Explicaste al menos 5 predicciones individuales (SHAP waterfall)
- [ ] Investigaste predicciones anómalas
- [ ] Verificaste interacciones con SHAP dependence
- [ ] Documentaste hallazgos clave
- [ ] Preparaste visualizaciones para stakeholders
- [ ] Mapeaste features encoded a valores originales

---

### 💡 Tips Finales

1. ✅ **Empieza simple** (Feature Importance) → profundiza (SHAP)
2. ✅ **Visual > Números**: Usa plots siempre que sea posible
3. ✅ **Samplea con datasets grandes**: SHAP en 1000-5000 registros es suficiente
4. ✅ **Itera**: Interpretabilidad es un proceso, no un pasoúnicos
5. ✅ **Cuenta historias**: Conecta insights con problemas de negocio
6. ✅ **Sé honesto**: Si no entiendes algo, invéstigalo

---

In [0]:
# Instalar dependencias
%pip install shap lime --quiet

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
import shap
import lime
import lime.lime_tabular

# Configurar visualizaciones
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
shap.initjs()

print("✅ Librerías importadas correctamente")

In [0]:
# Cargar datasets
ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

df_ventas = pd.read_csv(ruta_datos + 'ventas.csv')
df_clientes = pd.read_csv(ruta_datos + 'clientes.csv')

print("✅ Datasets cargados:")
print(f"   Ventas: {len(df_ventas):,} registros")
print(f"   Clientes: {len(df_clientes):,} registros")

# Preparar datos
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
df_ventas = df_ventas.sort_values('fecha').reset_index(drop=True)

# Features temporales
df_ventas['dia_semana'] = df_ventas['fecha'].dt.dayofweek
df_ventas['mes'] = df_ventas['fecha'].dt.month
df_ventas['es_fin_semana'] = df_ventas['dia_semana'].isin([5, 6]).astype(int)

# Merge con clientes
df = df_ventas.merge(df_clientes, on='cliente_id', how='left')
df['segmento_encoded'] = df['segmento'].astype('category').cat.codes

print("\n✅ Features temporales creadas")

In [0]:
print("="*80)
print("EJEMPLO COMPLETO: INTERPRETABILIDAD CON SHAP")
print("="*80)

# Preparar datos
df_ml = df[df['cliente_id'].notna()].copy()

features = ['sucursal_id', 'dia_semana', 'mes', 'es_fin_semana', 'segmento_encoded']
X = df_ml[features]
y = df_ml['total']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\n📊 Dataset: {len(X):,} registros, {len(features)} features")
print(f"   Train: {len(X_train):,}")
print(f"   Test: {len(X_test):,}")

# Entrenar modelo
print("\n1️⃣ Entrenando Random Forest...")
model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

from sklearn.metrics import mean_absolute_error, r2_score
mae = mean_absolute_error(y_test, model.predict(X_test))
r2 = r2_score(y_test, model.predict(X_test))
print(f"   MAE: ${mae:.2f}")
print(f"   R²: {r2:.3f}")

# Feature Importance
print("\n2️⃣ Feature Importance (RF):")
for feat, imp in sorted(zip(features, model.feature_importances_), key=lambda x: -x[1]):
    print(f"   {feat:20s}: {imp:.4f}")

# SHAP
print("\n3️⃣ Calculando SHAP values...")
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test[:500])  # Samplear para velocidad

print(f"   SHAP values shape: {shap_values.values.shape}")
print(f"   Base value: ${explainer.expected_value:.2f}")

# Summary plot
print("\n4️⃣ SHAP Summary Plot:")
shap.summary_plot(shap_values.values, X_test[:500], show=False)
plt.tight_layout()
plt.show()

# Waterfall para una predicción
print("\n5️⃣ SHAP Waterfall para una predicción:")
shap.plots.waterfall(shap_values[0])

print(f"\n" + "="*80)
print("✅ Ejemplo completado!")
print("="*80)

## ✅ Conclusiones

### 🎯 Resumen del Módulo

**Lo que aprendimos**:

1. ✅ **Por qué interpretabilidad** (confianza, compliance, debugging, bias)
2. ✅ **Interpretabilidad global vs. local** (entender modelo vs. predicción)
3. ✅ **Feature Importance** (rápido pero limitado)
4. ✅ **Permutation Importance** (más confiable)
5. ✅ **PDP e ICE** (visualizar efectos de features)
6. ✅ **SHAP** (método más robusto y completo)
7. ✅ **LIME** (alternativa local)
8. ✅ **Comparación** de métodos y cuándo usar cada uno

---

### 💡 Mensajes Clave

1. 🔑 **Interpretabilidad != Causalidad** - Asociación no implica causalidad
2. 🎯 **SHAP es el gold standard** - Más completo, robusto y rápido (para trees)
3. ⚠️ **No confiar solo en Feature Importance** - Validar con Permutation/SHAP
4. 📊 **Visual > Números** - Usar plots para comunicar
5. 🔄 **Combinar global + local** - Entendimiento completo requiere ambos
6. 💡 **Samplear con datos grandes** - 1000-5000 registros suficientes para SHAP

---

### 📦 Guía Rápida

**¿Qué método usar?**

- 🚀 **Exploración rápida** → Feature Importance
- ⚖️ **Importancia confiable** → Permutation Importance
- 📊 **Efecto de features** → PDP / ICE
- 🎯 **Explicaciones completas** (trees) → **SHAP TreeExplainer**
- 👤 **Explicaciones individuales** (no-trees) → LIME
- 🔍 **Debugging y validación** → Múltiples métodos

---

### 📚 Recursos Adicionales

- [SHAP Documentation](https://shap.readthedocs.io/)
- [Interpretable Machine Learning Book](https://christophm.github.io/interpretable-ml-book/)
- [LIME Paper](https://arxiv.org/abs/1602.04938)
- [SHAP Paper](https://arxiv.org/abs/1705.07874)

---

## 🎓 ¡Felicitaciones!

**Has completado el módulo de Interpretabilidad de Modelos.**

Ahora puedes:
- ✅ Explicar por qué un modelo hizo una predicción
- ✅ Identificar features más importantes (confiablemente)
- ✅ Visualizar efectos de features con PDP/ICE
- ✅ Usar SHAP para explicaciones globales y locales
- ✅ Comparar métodos y elegir el adecuado
- ✅ Detectar data leakage y bias
- ✅ Comunicar insights a stakeholders

**Próximo paso**: Notebook Práctico con ejercicios hands-on.

---

**Universidad del Aconcagua**  
**Laboratorio (Herramientas)**  
**Mendoza, Argentina**

## 8️⃣ LIME (Local Interpretable Model-agnostic Explanations)

### 📖 Concepto

**LIME** explica **predicciones individuales** aproximando el modelo complejo con un **modelo simple local**.

**Idea**:
1. Tomar una predicción a explicar
2. Generar **datos sintéticos** cerca de esa predicción
3. Predecir con el modelo original
4. Entrenar **modelo lineal simple** en esos datos
5. Usar el modelo lineal para explicar

**Analogía**: El modelo complejo es un **mapa 3D** complicado. LIME crea un **plano tangente** en un punto específico para entenderlo localmente.

---

### 💻 Instalación

```python
%pip install lime

import lime
import lime.lime_tabular
```

---

### 💻 Implementación

```python
import lime
import lime.lime_tabular
import numpy as np

# Entrenar modelo
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Crear explicador LIME
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X_train.columns.tolist(),
    mode='regression',  # 'classification' para clasificación
    random_state=42
)

# Explicar una predicción
i = 0  # Índice de la predicción a explicar
explanation = explainer.explain_instance(
    data_row=X_test.iloc[i].values,
    predict_fn=model.predict,
    num_features=5  # Top 5 features
)

# Visualizar
explanation.show_in_notebook()

# O como lista
print(f"Predicción: ${model.predict(X_test.iloc[[i]])[0]:.2f}")
print("\nContribuciones (LIME):")
for feat, weight in explanation.as_list():
    print(f"  {feat}: {weight:+.2f}")
```

---

### 📊 Visualización LIME

**Output típico**:
```
Predicción: $152.30
Intercept: $120.00

Contribuciones:
  dia_semana = 5        +$16.50
  zona_h3 = microcentro +$14.80
  mes = 1               -$3.20
  segmento = VIP        +$4.20
```

---

### ✅ Ventajas de LIME

1. ✅ **Model-agnostic**: Funciona con CUALQUIER modelo
2. ✅ **Intuitivo**: Modelo lineal fácil de entender
3. ✅ **Flexible**: Funciona con tablas, texto, imágenes
4. ✅ **Local**: Enfocado en explicar UNA predicción

---

### ❌ Desventajas de LIME

1. ❌ **Inestable**: Diferentes ejecuciones → diferentes explicaciones
2. ❌ **Muestreo**: Depende de cómo se generan datos sintéticos
3. ❌ **Lento**: Más lento que SHAP TreeExplainer
4. ❌ **Solo local**: No da visión global del modelo
5. ❌ **No suma**: Contribuciones no suman exactamente a la predicción

---

### ⚖️ LIME vs. SHAP

| Aspecto | LIME | SHAP |
|---------|------|------|
| **Scope** | Solo local | Global + Local |
| **Velocidad** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ (TreeExplainer) |
| **Estabilidad** | ❌ Inestable | ✅ Estable |
| **Sumas** | ❌ No | ✅ Sí |
| **Teoría** | Heurística | Shapley values |
| **Visualizaciones** | Básicas | ✅ Ricas |

**Conclusión**: 🎯 **SHAP es generalmente mejor** (más robusto, rápido para trees, mejor fundamentado).

---

### 📊 Cuándo Usar LIME

✅ **Usar cuando**:
- Modelo no es tree-based (ej: SVM, redes neuronales) Y SHAP KernelExplainer es muy lento
- Necesitas explicar **texto o imágenes** (LIME tiene soporte especial)
- Ya tienes pipeline con LIME

❌ **No usar cuando**:
- Tienes Random Forest/XGBoost → Usar SHAP TreeExplainer
- Necesitas explicaciones **globales** → Usar SHAP
- Necesitas **estabilidad** → Usar SHAP

---

## 9️⃣ Comparación Completa de Métodos

### 📊 Tabla Comparativa

| Método | Scope | Model-Agnostic | Velocidad | Estabilidad | Visualizaciones | Mejor Para |
|---------|-------|----------------|-----------|-------------|-----------------|------------|
| **Feature Importance** | Global | ❌ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ | Exploración rápida (trees) |
| **Permutation Importance** | Global | ✅ | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐ | Importancia confiable |
| **PDP** | Global | ✅ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | Efecto de 1 feature |
| **ICE** | Local | ✅ | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | Heterogeneidad |
| **SHAP** | Ambos | ✅ | ⭐⭐⭐⭐⭐* | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | **TODO** (más completo) |
| **LIME** | Local | ✅ | ⭐⭐⭐ | ⭐⭐ | ⭐⭐ | Texto, imágenes |

\* SHAP TreeExplainer es muy rápido; KernelExplainer es lento.

---

### ⚖️ Trade-offs

**Velocidad vs. Completitud**:
```
Feature Importance (rápido, básico)
  ↓
Permutation Importance
  ↓
PDP / ICE
  ↓
SHAP (completo, más lento)
  ↓
LIME (lento, inestable)
```

**Global vs. Local**:
```
Global                 Ambos              Local
  │                    │                  │
Feature Imp.          SHAP              LIME
Permutation                             ICE
PDP
```

---

### 🧩 Árbol de Decisión: ¿Qué Método Usar?

```
¿Tienes modelo tree-based (RF, XGBoost)?
│
├── SÍ → ¿Qué necesitas?
│   ├── Exploración rápida → Feature Importance
│   ├── Importancia confiable → Permutation Importance
│   ├── Efecto de features → PDP / ICE
│   └── Explicaciones completas → SHAP TreeExplainer 🎯
│
└── NO → ¿Qué necesitas?
    ├── Importancia global → Permutation Importance
    ├── Efecto de features → PDP / ICE
    ├── Explicaciones locales → SHAP KernelExplainer o LIME
    └── Texto/Imágenes → LIME
```

---

### 📊 Workflow Recomendado

**Fase 1: Exploración Rápida** (⏱️ 5 minutos)
1. Feature Importance (si tree-based)
2. Permutation Importance

**Fase 2: Análisis Global** (⏱️ 30 minutos)
3. PDP para top 5 features
4. SHAP Summary Plot
5. SHAP Dependence Plots para interacciones

**Fase 3: Explicaciones Locales** (⏱️ 15 minutos por caso)
6. SHAP Waterfall para casos específicos
7. SHAP Force plots para comparar predicciones

**Fase 4: Presentación** (⏱️ 1 hora)
8. Preparar visualizaciones SHAP
9. PDP para stakeholders
10. Waterfall para casos de negocio

---

### 💡 Recomendación General

🎯 **Para la mayoría de casos con Random Forest/XGBoost**:

1. ✅ **Usa SHAP** como método principal (TreeExplainer es rápido)
2. ✅ **Complementa con PDP** para visualizaciones simples
3. ✅ **Permutation Importance** para validar feature importance

🎯 **Para modelos no tree-based**:

1. ✅ **Permutation Importance** para importancia global
2. ✅ **PDP / ICE** para efectos de features
3. ✅ **SHAP KernelExplainer** si tiempo lo permite (lento)
4. ✅ **LIME** como alternativa más rápida

---

## 7️⃣ SHAP Values (SHapley Additive exPlanations)

### 🎯 ¿Qué es SHAP?

**SHAP** es el método de interpretabilidad **más robusto y completo** actualmente.

**Ventajas**:
1. ✅ **Global Y local**: Explica modelo completo + predicciones individuales
2. ✅ **Model-agnostic**: Funciona con cualquier modelo
3. ✅ **Fundamentado teóricamente**: Basado en Shapley values (teoría de juegos)
4. ✅ **Propiedades deseables**: Consistencia, simetría, dummy

**Pregunta que responde**:
> "Para esta predicción, ¿cuánto contribuyó cada feature?"

---

### 📚 Shapley Values: Intuición

**Analogía**: Repartir crédito en un equipo.

**Problema**: Tu equipo de fútbol ganó 3-1. ¿Cuánto contribuyó cada jugador?

**Shapley Value**: Valor promedio de un jugador **considerando todas las coaliciones posibles**.

**Ejemplo ML**:
```python
# Predicción: $150
# Baseline (sin features): $100
# Diferencia a explicar: +$50

# SHAP reparte los $50 entre features:
dia_viernes:    +$20  (40% del crédito)
zona_h3_centro: +$15  (30%)
cliente_VIP:    +$10  (20%)
mes_diciembre:  +$5   (10%)
-----------------
Total:          +$50  (100%)
```

🔑 **Propiedad clave**: SHAP values **suman** al total.

```
Predicción = Baseline + Σ(SHAP values)
$150 = $100 + ($20 + $15 + $10 + $5)
```

---

### 📊 SHAP Value Formula

**Definición matemática** (no te asustes, la librería lo calcula):

```
SHAPᵢ = Σ |S|! (N - |S| - 1)! / N! [f(S ∪ {i}) - f(S)]
      S
```

Donde:
- S = subconjunto de features
- i = feature de interés
- N = total de features
- f(S) = predicción con features en S

💡 **En palabras**: Promedio de **contribución marginal** de la feature sobre **todas las posibles coaliciones**.

---

### 💻 Instalación y Setup

```python
# Instalar SHAP
%pip install shap

import shap
import matplotlib.pyplot as plt

# Inicializar visualizaciones de SHAP
shap.initjs()
```

---

### 🔧 Tipos de Explicadores SHAP

SHAP tiene **diferentes explicadores** según el tipo de modelo:

| Explicador | Modelos | Velocidad | Precisión |
|------------|---------|-----------|------------|
| **TreeExplainer** | RF, XGBoost, LightGBM | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **LinearExplainer** | Regresión Lineal, Logística | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **KernelExplainer** | Cualquier modelo | ⭐ | ⭐⭐⭐⭐ |
| **DeepExplainer** | Redes Neuronales (TensorFlow/PyTorch) | ⭐⭐⭐ | ⭐⭐⭐⭐ |

💡 **Recomendación**: Usar **TreeExplainer** para RF/XGBoost (rápido y exacto).

---

### 💻 Implementación Básica

```python
import shap
from sklearn.ensemble import RandomForestRegressor

# Entrenar modelo
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Crear explicador SHAP
explainer = shap.TreeExplainer(model)

# Calcular SHAP values
shap_values = explainer(X_test)

print(f"SHAP values shape: {shap_values.values.shape}")
print(f"Base value: {explainer.expected_value:.2f}")
```

**Estructura de shap_values**:
```python
shap_values.values       # Array (N_samples, N_features)
shap_values.base_values  # Predicción baseline
shap_values.data         # Valores originales de features
```

---

### 📊 SHAP Value para Una Predicción

```python
# Seleccionar una predicción
i = 0  # Primera predicción de test

# SHAP values para esa predicción
shap_sample = shap_values[i]

print(f"Predicción: ${model.predict(X_test.iloc[[i]])[0]:.2f}")
print(f"Baseline: ${shap_sample.base_values:.2f}")
print(f"\nContribuciones por feature:")

for feat, val in zip(X_test.columns, shap_sample.values):
    sign = '+' if val >= 0 else ''
    print(f"  {feat:20s}: {sign}${val:.2f}")

print(f"\nSuma: ${shap_sample.base_values + shap_sample.values.sum():.2f}")
```

**Salida**:
```
Predicción: $152.30
Baseline: $120.00

Contribuciones por feature:
  dia_semana          : +$15.20
  zona_h3_encoded     : +$18.50
  mes                 : -$2.40
  segmento_encoded    : +$1.00

Suma: $152.30  # ✅ Suma exactamente!
```

---

## 7️⃣ SHAP Values - Visualizaciones

### 📊 1. Waterfall Plot (Explicación Local)

**Uso**: Explicar **UNA predicción específica**.

**Visualiza**: Cómo cada feature mueve la predicción desde baseline hasta el valor final.

```python
import shap

# Waterfall para una predicción
shap.plots.waterfall(shap_values[0])
```

**Interpretación**:
```
E[f(X)] = $120.00  (baseline)
  │
  ├─ dia_semana=5         +$15.20  ⬆️
  ├─ zona_h3=microcentro  +$18.50  ⬆️
  ├─ mes=1               -$2.40   ⬇️
  └─ segmento=VIP        +$1.00   ⬆️
  │
f(x) = $152.30  (predicción)
```

💡 **Uso**: Explicar predicciones a usuarios finales.

---

### 📊 2. Force Plot (Explicación Local)

**Similar a waterfall** pero visualización horizontal.

```python
# Force plot para una predicción
shap.plots.force(shap_values[0])
```

**Force plot para múltiples predicciones**:
```python
# Apilar múltiples force plots
shap.plots.force(shap_values[:100])  # Primeras 100 predicciones
```

---

### 📊 3. Summary Plot (Interpretabilidad Global)

**Uso**: Visualizar **importancia global** + **distribución** de SHAP values.

```python
# Summary plot (beeswarm)
shap.summary_plot(shap_values.values, X_test)
```

**Interpretación**:
- **Eje Y**: Features ordenadas por importancia
- **Eje X**: SHAP value (impacto en predicción)
- **Color**: Valor de la feature (rojo=alto, azul=bajo)
- **Puntos**: Cada punto es una predicción

**Ejemplo**:
```
Feature         SHAP value
            -20  -10   0  +10  +20
zona_h3     ●●●●|●●●●●●  (rojo a derecha → zonas buenas aumentan ventas)
dia_semana  ●●●|●●●●  (disperso → efecto depende del día)
mes         ●●|●●●  (centrado → efecto variable)
```

💡 **Insights**:
- Features en **top** son más importantes
- **Dispersión horizontal** → efecto variable
- **Color** muestra dirección del efecto

---

### 📊 4. Bar Plot (Feature Importance Global)

**Uso**: Importancia global (similar a feature importance).

```python
# Bar plot (importancia promedio absoluta)
shap.summary_plot(shap_values.values, X_test, plot_type='bar')
```

**Interpretación**: Muestra **|SHAP value| promedio** por feature.

---

### 📊 5. Dependence Plot (Relación Feature-Predicción)

**Uso**: Ver cómo **una feature** afecta predicciones (similar a PDP pero con SHAP).

```python
# Dependence plot para una feature
shap.dependence_plot(
    'dia_semana',  # Feature principal
    shap_values.values,
    X_test,
    interaction_index='mes'  # Color por interacción con otra feature
)
```

**Interpretación**:
- **Eje X**: Valor de feature
- **Eje Y**: SHAP value (impacto)
- **Color**: Interacción con otra feature

**Ejemplo**: 
```
SHAP value
   │
+20│      🔴🔴  (viernes + diciembre)
   │    🔵
+10│  🔵    🔵
   │🔵      🔴
  0│🔴    🔵
   │
 -10│🔵
   └──────────────> dia_semana
    Lun       Vie      Dom
    🔵=verano 🔴=invierno
```

💡 **Insight**: Viernes tiene mayor impacto en invierno que en verano.

---

### 📊 6. Decision Plot (Trayectoria de Predicción)

**Uso**: Ver **cómo se acumula** la predicción feature por feature.

```python
# Decision plot
shap.decision_plot(
    explainer.expected_value,
    shap_values.values[:10],  # Primeras 10 predicciones
    X_test.iloc[:10]
)
```

---

### ⚖️ Qué Visualización Usar

| Objetivo | Visualización |
|----------|----------------|
| **Explicar 1 predicción** | Waterfall o Force |
| **Importancia global** | Summary (beeswarm) o Bar |
| **Efecto de 1 feature** | Dependence Plot |
| **Comparar predicciones** | Force (múltiples) o Decision |
| **Interacciones** | Dependence (con color) |

---

## 5️⃣ Partial Dependence Plots (PDP)

### 📖 Concepto

**PDP** muestra **cómo afecta una feature** a las predicciones del modelo, **promediando** sobre todas las demás features.

**Pregunta**: ¿Cómo cambia la predicción cuando cambio esta feature, **manteniendo todo lo demás constante**?

**Proceso**:
1. Seleccionar feature de interés (ej: `dia_semana`)
2. Para cada valor posible de `dia_semana` (0-6):
   - Crear dataset donde TODOS los registros tienen `dia_semana = 0`
   - Predecir y promediar
   - Crear dataset donde TODOS los registros tienen `dia_semana = 1`
   - Predecir y promediar
   - ... repetir para 2, 3, 4, 5, 6
3. Graficar: `dia_semana` vs. predicción promedio

---

### 💻 Implementación

```python
from sklearn.inspection import PartialDependenceDisplay
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

# Entrenar modelo
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# PDP para una feature
fig, ax = plt.subplots(figsize=(10, 6))
PartialDependenceDisplay.from_estimator(
    model, X_train,
    features=['dia_semana'],  # Feature de interés
    ax=ax
)
plt.title('PDP: Efecto de Día de la Semana en Ventas')
plt.show()
```

**PDP para múltiples features**:
```python
# PDP para top 4 features
fig, ax = plt.subplots(figsize=(14, 10))
PartialDependenceDisplay.from_estimator(
    model, X_train,
    features=['dia_semana', 'mes', 'zona_h3_encoded', 'segmento_encoded'],
    ax=ax
)
plt.tight_layout()
plt.show()
```

---

### 📊 Interpretación de PDP

**Ejemplo**: PDP de `dia_semana` para ventas

```
Predicción
   $
   │
150│             ┌───┐
   │             │   │
140│         ┌───┤   ├──┐
   │     ┌───┤   │   │  │
130│     │   │   │   │  │
   │─────┤   │   │   │  ├─────
120│     │   └───┘   └──┘
   └───────────────────────────> Día
    Lun Mar Mie Jue Vie Sab Dom
```

💡 **Insights**:
- Lunes: ventas bajas ($120)
- Viernes: ventas altas ($150)
- Fin de semana: caída ($140)

---

### ✅ Ventajas

1. ✅ **Visual e intuitivo**: Fácil de explicar
2. ✅ **Model-agnostic**: Funciona con cualquier modelo
3. ✅ **Muestra relación**: Lineal, no lineal, monotonicity
4. ✅ **Causal (parcial)**: Efecto de cambiar UNA feature

---

### ❌ Desventajas

#### 1. **Asume Independencia de Features**

⚠️ **Problema**: PDP crea combinaciones **imposibles** de features.

**Ejemplo**:
```python
# Dataset:
# - Zona H3 A → siempre sucursal 1
# - Zona H3 B → siempre sucursal 2

# PDP de zona H3:
# Crea combinaciones imposibles:
# - Zona H3 A + sucursal 2  → Nunca ocurre en realidad!
```

💡 **Solución**: Usar **ICE plots** (Individual Conditional Expectation).

#### 2. **Oculta Heterogeneidad**

```python
# PDP muestra promedio:
# - 50% clientes: ventas suben con precio
# - 50% clientes: ventas bajan con precio
# PDP promedio: Sin efecto (falso!)
```

---

### 🔄 PDP de 2 Features (2D)

**PDP 2D** muestra **interacción** entre dos features.

```python
from sklearn.inspection import PartialDependenceDisplay

fig, ax = plt.subplots(figsize=(10, 8))
PartialDependenceDisplay.from_estimator(
    model, X_train,
    features=[('dia_semana', 'mes')],  # Tuple para 2D
    ax=ax
)
plt.title('PDP 2D: Interacción Día-Mes')
plt.show()
```

**Interpretación**: Heatmap muestra predicción para cada combinación.

---

### 📊 Cuándo Usar PDP

✅ **Usar cuando**:
- Quieres entender **efecto global** de una feature
- Presentar a stakeholders (visual)
- Detectar **relaciones no lineales**
- Comparar modelos

❌ **No usar cuando**:
- Features están **muy correlacionadas** (usar ICE)
- Quieres explicar **predicciones individuales** (usar SHAP/LIME)

---

## 6️⃣ Individual Conditional Expectation (ICE)

### 📖 Concepto

**ICE plots** son como PDP pero **sin promediar** → muestran una línea **por cada registro**.

**Diferencia con PDP**:
- **PDP**: Una línea (promedio de todos los registros)
- **ICE**: N líneas (una por registro)

**Ventaja**: Revela **heterogeneidad** (diferentes efectos en diferentes registros).

---

### 💻 Implementación

```python
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt

# ICE plot
fig, ax = plt.subplots(figsize=(10, 6))
PartialDependenceDisplay.from_estimator(
    model, X_train,
    features=['dia_semana'],
    kind='individual',  # 🔑 ICE en lugar de PDP
    ax=ax
)
plt.title('ICE: Efecto de Día de Semana (por registro)')
plt.show()
```

**ICE + PDP juntos**:
```python
fig, ax = plt.subplots(figsize=(10, 6))
PartialDependenceDisplay.from_estimator(
    model, X_train,
    features=['dia_semana'],
    kind='both',  # 🔑 ICE (líneas) + PDP (promedio)
    ax=ax
)
plt.title('ICE + PDP: Día de Semana')
plt.show()
```

---

### 📊 Interpretación de ICE

**Ejemplo 1: Efecto Homogéneo**
```
Predicción
   $
   │      Todas las líneas
150│      van en la misma
   │      dirección →
140│     ┌─────────┐
   │    /          \
130│   /            \
   │  /              \
120│ /                \
   └────────────────────> Día
    Lun       Vie      Dom
```
💡 **Efecto consistente**: Viernes aumenta ventas para TODOS.

**Ejemplo 2: Efecto Heterogéneo**
```
Predicción
   $
   │  Líneas van en
150│  direcciones opuestas!
   │    ┌──────
140│   /
   │  /   \
130│ /     \
   │/       \
120│         ╰─────
   └──────────────────> Día
    Lun       Vie      Dom
```
💡 **Efecto heterogéneo**: 
- 50% clientes: ventas suben los viernes
- 50% clientes: ventas bajan los viernes
- PDP mostraría promedio (sin efecto) → **engañoso**!

---

### ⚖️ PDP vs. ICE

| Aspecto | PDP | ICE |
|---------|-----|-----|
| **Visualización** | 1 línea (promedio) | N líneas (individuales) |
| **Muestra heterogeneidad** | ❌ No | ✅ Sí |
| **Fácil de interpretar** | ✅ Sí | ❌ Puede ser confuso con muchos datos |
| **Velocidad** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ (más plots) |

💡 **Recomendación**: Usar **ambos juntos** (kind='both').

---

### 📊 Cuándo Usar ICE

✅ **Usar cuando**:
- Sospechas **interacciones** complejas
- Quieres ver **variabilidad** individual
- Dataset tiene **subgrupos** diferentes

⚠️ **Cuidado con**:
- Datasets muy grandes (demasiadas líneas) → samplear
- Features discretas (líneas superpuestas)

---

## 3️⃣ Feature Importance (Tree-based)

### 📖 Concepto

**Feature Importance** mide **cuánto contribuye cada feature** a reducir la impureza en árboles de decisión.

**Cálculo** (para Random Forest):
1. Para cada árbol, calcular reducción de impureza por feature
2. Promediar sobre todos los árboles
3. Normalizar a suma = 1

**Impureza**: Gini (clasificación) o MSE (regresión)

---

### 💻 Implementación

```python
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import matplotlib.pyplot as plt

# Entrenar modelo
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importances)

# Visualizar
plt.figure(figsize=(10, 6))
plt.barh(importances['feature'][:10], importances['importance'][:10])
plt.xlabel('Importancia')
plt.title('Top 10 Features más Importantes')
plt.gca().invert_yaxis()
plt.show()
```

---

### ✅ Ventajas

1. ✅ **Rápido**: Calculado durante entrenamiento
2. ✅ **Fácil de entender**: Un número por feature
3. ✅ **Incorporado**: Built-in en sklearn

---

### ❌ Desventajas y Limitaciones

#### 1. **Sesgo con Features de Alta Cardinalidad**

```python
# Feature con muchos valores únicos tiene ventaja injusta
df['id_transaccion']  # 10,000 valores únicos → Alta importancia (falsa)
df['dia_semana']      # 7 valores → Baja importancia (subestimada)
```

⚠️ **Problema**: Features con más valores tienen más oportunidades de split.

#### 2. **Sesgo con Features Correlacionadas**

```python
# Si feature_A y feature_B están correlacionadas:
# - Importancia se "reparte" entre ambas
# - Una puede dominar arbitrariamente
```

#### 3. **Solo para Modelos Tree-based**

❌ No funciona con:
- Regresión lineal
- SVM
- Redes neuronales
- KNN

#### 4. **No Captura Dirección**

```python
# Feature importance dice: "mes" es importante (30%)
# Pero NO dice:
# - ¿Más ventas en verano o invierno?
# - ¿Relación lineal o no lineal?
```

---

### 📊 Ejemplo: Panadería

```python
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Cargar datos
df = pd.read_csv('ventas.csv')
X = df[['sucursal_id', 'dia_semana', 'mes', 'zona_h3_encoded', 'segmento_encoded']]
y = df['total']

# Entrenar
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# Feature importance
for feat, imp in zip(X.columns, model.feature_importances_):
    print(f"{feat:20s}: {imp:.4f}  {'=' * int(imp * 100)}")
```

**Salida**:
```
zona_h3_encoded     : 0.3521  ===================================
dia_semana          : 0.2845  ============================
mes                 : 0.1832  ==================
segmento_encoded    : 0.1204  ============
sucursal_id         : 0.0598  =====
```

💡 **Interpretación**: La zona H3 es la feature más importante (35%), seguida del día de la semana (28%).

---

### 🔑 Mensaje Clave

⚠️ **Feature Importance es útil como primer paso**, pero tiene **limitaciones**.

✅ **Mejor**: Combinar con **Permutation Importance** (más confiable).

---

## 4️⃣ Permutation Importance

### 📖 Concepto

**Permutation Importance** mide **cuánto empeora** el rendimiento cuando se **permutan (mezclan)** los valores de una feature.

**Proceso**:
1. Entrenar modelo y calcular rendimiento baseline
2. Para cada feature:
   - **Permutar** sus valores aleatoriamente
   - Predecir con feature permutada
   - Calcular nuevo rendimiento
   - Importancia = baseline - nuevo rendimiento
3. Repetir N veces (ej: 10) y promediar

**Intuición**: Si permutar una feature **empeora mucho** el modelo → feature es **importante**.

---

### 💻 Implementación

```python
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Permutation importance
result = permutation_importance(
    model, X_test, y_test,
    n_repeats=10,  # Repeticiones
    random_state=42,
    n_jobs=-1
)

# Resultados
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': result.importances_mean,
    'std': result.importances_std
}).sort_values('importance', ascending=False)

print(importances)

# Visualizar con error bars
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(
    importances['feature'][:10],
    importances['importance'][:10],
    xerr=importances['std'][:10]  # Error bars
)
plt.xlabel('Permutation Importance')
plt.title('Top 10 Features por Permutation Importance')
plt.gca().invert_yaxis()
plt.show()
```

---

### ✅ Ventajas

1. ✅ **Model-agnostic**: Funciona con **cualquier modelo**
2. ✅ **Más confiable** que RF importance (sin sesgo de cardinalidad)
3. ✅ **Maneja correlaciones** mejor
4. ✅ **Incluye varianza** (error bars)

---

### ❌ Desventajas

1. ❌ **Más lento**: Requiere N × M predicciones (N features, M repeticiones)
2. ❌ **Requiere test set**: No usar train (sobreestima importancia)
3. ❌ **Puede ser inestable**: Con features muy correlacionadas

---

### ⚖️ RF Importance vs. Permutation Importance

**Comparación**:

| Aspecto | RF Importance | Permutation Importance |
|---------|---------------|-------------------------|
| **Velocidad** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ |
| **Confiabilidad** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Sesgo cardinalidad** | ❌ Sí | ✅ No |
| **Model-agnostic** | ❌ No | ✅ Sí |
| **Incertidumbre** | ❌ No | ✅ Sí (std) |

**Ejemplo de Diferencia**:

```python
# Dataset con feature correlacionadas
df['peso_kg'] = ...   # Peso en kilogramos
df['peso_lb'] = df['peso_kg'] * 2.205  # Peso en libras (perfectamente correlacionado)

# RF Importance:
peso_kg: 0.15
peso_lb: 0.35  # ⚠️ Sobrestimado (alta cardinalidad por decimales)

# Permutation Importance:
peso_kg: 0.25
peso_lb: 0.25  # ✅ Similar (correctamente)
```

---

### 📊 Cuándo Usar

**Usa RF Importance cuando**:
- Exploración rápida
- Solo usas Random Forest/XGBoost
- Dataset grande (velocidad crítica)

**Usa Permutation Importance cuando**:
- Quieres **máxima confiabilidad**
- Tienes features **correlacionadas**
- Usas modelos **no tree-based**
- Presentación a stakeholders

💡 **Recomendación**: **Siempre usar Permutation** para análisis final.

---

# 🔍 Interpretabilidad de Modelos de Machine Learning
## Material Complementario - Laboratorio (Herramientas)
### Universidad del Aconcagua - Mendoza, Argentina

---

### 🎯 Objetivos de Aprendizaje

1. Comprender **por qué la interpretabilidad** es crucial
2. Diferenciar entre **interpretabilidad global vs. local**
3. Dominar **Feature Importance** y Permutation Importance
4. Aplicar **Partial Dependence Plots (PDP)** e **ICE**
5. Usar **SHAP values** para explicaciones robustas
6. Implementar **LIME** para explicaciones locales
7. Comparar métodos y elegir el adecuado
8. Aplicar a **features H3 y temporales**

### 📁 Contenido

1. ¿Por qué Interpretabilidad?
2. Interpretabilidad Global vs. Local
3. Feature Importance (Tree-based)
4. Permutation Importance
5. Partial Dependence Plots (PDP)
6. Individual Conditional Expectation (ICE)
7. SHAP Values (SHapley Additive exPlanations)
8. LIME (Local Interpretable Model-agnostic Explanations)
9. Casos Prácticos con H3
10. Comparación de Métodos
11. Mejores Prácticas

### ⏱️ Duración Estimada: 2-3 horas

---

## 1️⃣ ¿Por qué Necesitamos Interpretabilidad?

### 🎭 El Problema: Modelos "Black Box"

**Escenario típico**:
```python
model = RandomForestRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# ¿Pero CÓMO llegó a esta predicción?
# ¿QUÉ features fueron importantes?
# ¿POR QUÉ predijo $150 y no $100?
```

❌ **Problema**: El modelo funciona pero **no sabemos por qué**.

---

### 💡 ¿Qué es Interpretabilidad?

**Interpretabilidad** = Capacidad de **explicar** o **entender** las decisiones de un modelo de ML.

**Dos dimensiones**:

1. **Interpretabilidad Global**: ¿Cómo funciona el modelo en general?
   - ¿Qué features son más importantes?
   - ¿Cómo afecta cada feature a las predicciones?

2. **Interpretabilidad Local**: ¿Por qué el modelo hizo ESTA predicción específica?
   - Para este cliente, ¿por qué predijo $150?
   - ¿Qué cambiaría para obtener una predicción diferente?

---

### 🎯 ¿Por qué es Importante?

#### 1. **Confianza y Adopción**

```python
# Stakeholder: "¿Por qué debo confiar en este modelo?"
# Tú: "Porque tiene 95% de accuracy"
# Stakeholder: "Pero ¿CÓMO funciona?"
```

✅ **Con interpretabilidad**: "El modelo usa principalmente el historial de compras y la ubicación geográfica. Para este cliente, su alta frecuencia de compra y zona H3 urbana aumentan la predicción en $30."

#### 2. **Cumplimiento Legal y Regulatorio**

📜 **GDPR (Europa)**: "Derecho a explicación" de decisiones automatizadas.
📜 **Fair Lending (USA)**: Explicar decisiones de crédito.
📜 **BCRA (Argentina)**: Transparencia en modelos de riesgo.

#### 3. **Debugging y Mejora del Modelo**

```python
# Feature importance revela:
# - "ID de transacción" tiene alta importancia → 🚨 Data leakage!
# - "Mes" no es importante → 💡 Estacionalidad no capturada
```

#### 4. **Detección de Bias**

```python
# SHAP muestra:
# - Género afecta predicción salarial → 🚨 Sesgo discriminatorio
# - Código postal (proxy de raza) predice crédito → 🚨 Red-lining
```

#### 5. **Conocimiento del Negocio**

💡 **Descubrir patrones** que el negocio no conocía:
- "Clientes que compran pan los lunes tienen 2x más probabilidad de volver"
- "La zona H3 del microcentro tiene ventas 30% mayores de 7-9am"

#### 6. **Acción y Recomendaciones**

```python
# Para aumentar la predicción de ventas:
# SHAP dice: "Aumentar inventario en zona H3 X los viernes +15%"
# LIME dice: "Este cliente compraría más si le ofrecemos descuento en medialunas"
```

---

### ⚖️ Trade-off: Accuracy vs. Interpretabilidad

**Espectro de modelos**:

```
Interpretables                       Black Box
│                                        │
│  Regresión Lineal                      │
│  │                                     │
│  │  Regresión Logística                │
│  │  │                                  │
│  │  │  Árbol de Decisión               │
│  │  │  │                               │
│  │  │  │  Random Forest                │
│  │  │  │  │                            │
│  │  │  │  │  XGBoost                   │
│  │  │  │  │  │                         │
│  │  │  │  │  │  Redes Neuronales       │
│  │  │  │  │  │  │                      │
⭐⭐⭐  ⭐⭐  ⭐  ⭐  ⭐  ⭐  Accuracy           
```

**Dilemma**:
- Modelos simples (regresión lineal) → **Interpretables** pero **menos precisos**
- Modelos complejos (redes neuronales) → **Más precisos** pero **menos interpretables**

🎯 **Solución**: Usar **técnicas de interpretabilidad** (SHAP, LIME, PDP) para explicar modelos complejos.

---

### 📈 Ejemplo Motivador

**Caso**: Modelo predice ventas de panadería.

**Sin interpretabilidad**:
```
Predicción: $152.30
```

**Con interpretabilidad (SHAP)**:
```
Predicción Base: $120.00
+ Día viernes:        +$15.00
+ Zona H3 microcentro: +$20.00
+ Cliente frecuente:   +$8.00
- Mes enero (verano):  -$10.70
= Predicción Final:    $152.30
```

💡 **Ahora sabemos**:
- **Por qué** la predicción es alta
- **Qué features** contribuyeron más
- **Cómo cambiar** la predicción (ej: mover a otra zona)

---

### 📊 Tipos de Preguntas que Podemos Responder

**Interpretabilidad Global**:
1. ¿Cuáles son las 5 features más importantes?
2. ¿Cómo afecta "día de la semana" a las predicciones?
3. ¿Qué relación hay entre "zona H3" y "ventas"?

**Interpretabilidad Local**:
1. Para **este cliente**, ¿por qué predijimos $150?
2. ¿Qué cambiaría para que la predicción sea $200?
3. ¿Es esta predicción confiable?

---

## 2️⃣ Interpretabilidad Global vs. Local

### 🌍 Interpretabilidad Global

**Pregunta**: ¿Cómo funciona el modelo **en general**?

**Métodos**:
- Feature Importance
- Permutation Importance
- Partial Dependence Plots (PDP)
- SHAP Global (promedios)

**Ejemplo**: "El modelo usa principalmente 'zona H3' (30%) y 'día de semana' (25%) para predecir ventas."

**Cuándo usar**:
- Entender el modelo completo
- Comunicar a stakeholders
- Detectar data leakage
- Feature selection

---

### 🎯 Interpretabilidad Local

**Pregunta**: ¿Por qué el modelo hizo **esta predicción específica**?

**Métodos**:
- SHAP Local (valores individuales)
- LIME
- Individual Conditional Expectation (ICE)

**Ejemplo**: "Para el cliente #12345, la predicción de $150 se debe a: zona H3 microcentro (+$20), viernes (+$15), cliente frecuente (+$8)."

**Cuándo usar**:
- Explicar predicciones individuales
- Debugging de casos anómalos
- Compliance (explicar decisiones)
- Recomendaciones personalizadas

---

### 📐 Comparación Visual

**Interpretabilidad Global**:
```
🌍 Vista del bosque completo

   Feature Importance:
   │
   ├── zona_h3        ███████████████ 30%
   ├── dia_semana     ████████████ 25%
   ├── mes            ████████ 20%
   └── segmento       █████ 15%
```

**Interpretabilidad Local**:
```
🎯 Vista de un árbol específico

   Cliente #12345 - Predicción: $152.30
   │
   ├── Base:              $120.00
   ├── zona_h3:           +$20.00  ⬆️
   ├── dia_semana:        +$15.00  ⬆️
   ├── cliente_frecuente: +$8.00   ⬆️
   └── mes:               -$10.70  ⬇️
```

---

### 📊 Matriz de Métodos

| Método | Global | Local | Model-Agnostic | Velocidad |
|---------|--------|-------|----------------|------------|
| **Feature Importance** | ✅ | ❌ | ❌ (solo trees) | ⭐⭐⭐⭐⭐ |
| **Permutation Importance** | ✅ | ❌ | ✅ | ⭐⭐⭐ |
| **PDP** | ✅ | ❌ | ✅ | ⭐⭐⭐ |
| **ICE** | ❌ | ✅ | ✅ | ⭐⭐ |
| **SHAP** | ✅ | ✅ | ✅ | ⭐⭐ |
| **LIME** | ❌ | ✅ | ✅ | ⭐⭐⭐ |

**Model-Agnostic** = Funciona con cualquier modelo (no solo árboles)

---

### 🧠 Cuándo Usar Cada Tipo

**Usa Interpretabilidad Global cuando**:
- 📊 Presentas el modelo a stakeholders
- 🔍 Debuggeas el modelo (detectar leakage)
- ⚖️ Comparas features (feature selection)
- 📚 Documentas el modelo para producción

**Usa Interpretabilidad Local cuando**:
- 👤 Explicas una predicción a un usuario
- 🚨 Investigas casos anómalos (outliers)
- ⚖️ Cumples regulaciones (right to explanation)
- 🎯 Generas recomendaciones personalizadas

**Usa AMBAS cuando**:
- 🎯 Quieres entendimiento completo del modelo
- 📈 Presentación a negocio (global) + casos de uso (local)

---